# Stylometric SLM — mT5 fine-tune

Fine-tunes `google/mt5-base` on the multilingual authorship attribution
task. Reads `HF_TOKEN` from Colab secrets (or env), reads the corpus from
the Hugging Face dataset repo, runs a full fine-tune, evaluates on the
held-out split, pushes the fine-tuned model back to Hub.

Hardware: free Colab T4 (15GB VRAM) is sufficient for mT5-base full FT.
Training time: ~30-45 minutes for 12k examples, 5 epochs.

**Before running:**
1. Set `HF_TOKEN` in Colab secrets (key icon in left sidebar)
2. Set `HF_DATASET_REPO` and `HF_MODEL_REPO` in the Config cell below
3. Make sure the dataset is already uploaded (run `scripts/push_to_hf.py` first)

In [ ]:
# === Config ===
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')  # set in Colab secrets panel
assert HF_TOKEN, "HF_TOKEN missing — add it in Colab secrets panel"

# Repo targets — change to your own username
HF_USERNAME = "your-username"  # <-- change this
HF_DATASET_REPO = f"{HF_USERNAME}/stylometric-slm-corpus"
HF_MODEL_REPO = f"{HF_USERNAME}/stylometric-slm-mt5"

BASE_MODEL = "google/mt5-base"
EPOCHS = 5
BATCH_SIZE = 8
GRAD_ACCUM = 4  # effective batch = 32
LR = 3e-5
MAX_INPUT_LEN = 1024
MAX_TARGET_LEN = 32

os.environ['HF_TOKEN'] = HF_TOKEN

In [ ]:
# === Install ===
!pip install -q "transformers>=4.45" "datasets>=2.20" "sentencepiece>=0.2" accelerate

In [ ]:
# === Load dataset ===
from datasets import load_dataset

ds = load_dataset(HF_DATASET_REPO, token=HF_TOKEN)
print(f"train: {len(ds['train'])} examples")
print(f"eval:  {len(ds['eval'])} examples")
print(f"languages: {sorted(set(ds['train']['language']))}")
print(f"authors:   {sorted(set(ds['train']['author']))}")

In [ ]:
# === Load tokenizer + model ===
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

print(f"model params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# === Preprocess: text-to-text format ===
# Input:  "classify authorship: <passage>"
# Target: "<author>"
#
# mT5 is encoder-decoder; we frame classification as text generation.
# This is the canonical T5 approach for classification tasks.

PREFIX = "classify authorship: "

def preprocess(batch):
    inputs = [PREFIX + t for t in batch["text"]]
    targets = batch["author"]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = ds["train"].map(preprocess, batched=True, remove_columns=ds["train"].column_names)
eval_ds = ds["eval"].map(preprocess, batched=True, remove_columns=ds["eval"].column_names)
print(f"preprocessed train: {len(train_ds)}, eval: {len(eval_ds)}")

In [ ]:
# === Training ===
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = TrainingArguments(
    output_dir="/content/mt5-stylometric",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.05,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="epoch",
    bf16=True,
    push_to_hub=True,
    hub_model_id=HF_MODEL_REPO,
    hub_token=HF_TOKEN,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
# === Save + push final model ===
trainer.push_to_hub(commit_message="trained for 5 epochs on multilingual authorship attribution")
print(f"pushed to {HF_MODEL_REPO}")

In [ ]:
# === Quick eval: accuracy per author per language ===
import json
from collections import defaultdict

correct = defaultdict(lambda: defaultdict(int))
total = defaultdict(lambda: defaultdict(int))

model.eval()
with torch.no_grad():
    for example in ds["eval"]:
        inp = tokenizer(
            PREFIX + example["text"],
            return_tensors="pt",
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(model.device)
        out = model.generate(**inp, max_length=MAX_TARGET_LEN)
        pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        gold = example["author"]
        lang = example["language"]
        total[lang][gold] += 1
        if pred == gold:
            correct[lang][gold] += 1

print("accuracy per author per language:")
for lang in sorted(total.keys()):
    for author in sorted(total[lang].keys()):
        c = correct[lang][author]
        t = total[lang][author]
        acc = c / t if t else 0.0
        print(f"  {lang}/{author:<22s} {acc:.3f}  ({c}/{t})")

    overall_c = sum(correct[lang].values())
    overall_t = sum(total[lang].values())
    print(f"  {lang}/<overall>               {overall_c/overall_t:.3f}  ({overall_c}/{overall_t})")
    print()